[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/03-virtual-envs.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/03-virtual-envs.ipynb)

# Module 8 Lesson 3 — Virtual Environments & Dependency Management

**Module 8: Best Practices & Real-World Python** | Estimated time: 30 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Explain why virtual environments are essential for Python development
- Create and use virtual environments with the built-in `venv` module
- Manage dependencies with `pip`, `requirements.txt`, and `requirements.in`
- Use **pip-tools** to compile pinned requirements from abstract requirements
- Understand **Poetry** for modern dependency management and packaging
- Describe the structure and purpose of `pyproject.toml` and lock files
- Compare `venv`, `conda`, and `Poetry` for different use cases

In [ ]:
import sys
import subprocess

# Check current Python environment
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Pip version:", subprocess.getoutput("pip --version"))

## Why Virtual Environments?

Without a virtual environment, every project shares the same global Python installation. This causes:

- **Dependency conflicts**: Project A needs `requests==2.28`, Project B needs `requests==2.31`
- **Version pollution**: Old packages accumulate and break new projects
- **Reproducibility issues**: `pip install` on another machine might get different versions

A virtual environment is a self-contained Python installation for a single project.

```
Global Python 3.11
├── requests 2.28   ← needed by old-project
└── ??? conflict!

venv for project-a/     venv for project-b/
└── requests 2.28        └── requests 2.31
```

In [ ]:
# Show venv commands (we explain each step even though Colab has its own env)
venv_commands = """
# ── CREATING A VIRTUAL ENVIRONMENT ───────────────────────────────────────────

# Create a venv in a folder called '.venv' (convention)
python -m venv .venv

# OR: use a specific Python version
python3.11 -m venv .venv

# ── ACTIVATING THE ENVIRONMENT ───────────────────────────────────────────────

# macOS / Linux
source .venv/bin/activate

# Windows (Command Prompt)
.venv\\Scripts\\activate.bat

# Windows (PowerShell)
.venv\\Scripts\\Activate.ps1

# ── USING THE ENVIRONMENT ────────────────────────────────────────────────────

# Once activated, 'python' and 'pip' point to the venv
which python          # → /your/project/.venv/bin/python
pip install requests  # installed only in this venv

# ── DEACTIVATING ─────────────────────────────────────────────────────────────

deactivate
"""
print(venv_commands)

## pip — The Package Installer

Core `pip` commands every developer needs to know:

In [ ]:
# pip commands reference
pip_commands = """
# Install a package
pip install requests

# Install a specific version
pip install requests==2.31.0

# Install a minimum version
pip install "requests>=2.28,<3"

# Install from requirements file
pip install -r requirements.txt

# List installed packages
pip list
pip list --outdated     # show packages with newer versions

# Show package info
pip show requests

# Uninstall
pip uninstall requests

# Freeze current environment to a file
pip freeze > requirements.txt

# Upgrade pip itself
pip install --upgrade pip
"""
print(pip_commands)

# Actually list packages in this environment
print("=== Currently installed packages ===")
!pip list | head -20

## requirements.txt vs requirements.in

`pip freeze` produces a *fully pinned* `requirements.txt` with every transitive dependency. This can be fragile — a minor version bump in a transitive dep breaks the whole install.

The better workflow:
1. `requirements.in` — list only **direct** dependencies with loose constraints
2. `pip-compile` — resolve and pin everything into `requirements.txt`

In [ ]:
%%writefile requirements.in
# Direct dependencies only — versions are abstract/loose
requests>=2.28
fastapi>=0.100
pydantic>=2.0
uvicorn[standard]>=0.23
python-dotenv>=1.0

In [ ]:
# Install pip-tools
!pip install pip-tools --quiet

# Compile requirements.in → requirements.txt (fully pinned)
!pip-compile requirements.in --output-file requirements.txt --quiet

print("=== Generated requirements.txt ===")
!head -30 requirements.txt

In [ ]:
# Also create dev requirements
%%writefile requirements-dev.in
-r requirements.in   # include production deps

# Dev-only tools
pytest>=7.4
pytest-cov>=4.1
black>=23.0
flake8>=6.0
mypy>=1.5
isort>=5.12

In [ ]:
!pip-compile requirements-dev.in --output-file requirements-dev.txt --quiet

print("=== Generated requirements-dev.txt (first 20 lines) ===")
!head -20 requirements-dev.txt

print("\n=== pip-sync installs EXACTLY what is pinned ===")
# pip-sync removes packages not in the file, ensuring a clean env
# (we don't actually run this in Colab to avoid breaking the environment)
pip_sync_info = """pip-sync requirements.txt            # production
pip-sync requirements.txt requirements-dev.txt  # dev"""
print(pip_sync_info)

## Poetry — Modern Dependency Management

Poetry is an all-in-one tool that handles virtual environments, dependency resolution, and publishing to PyPI.

In [ ]:
# Install Poetry
!pip install poetry --quiet
!poetry --version

In [ ]:
# Poetry command reference
poetry_commands = """
# ── STARTING A NEW PROJECT ────────────────────────────────────────────────────

poetry new myproject          # scaffold new project
poetry init                   # interactively create pyproject.toml

# ── ADDING DEPENDENCIES ───────────────────────────────────────────────────────

poetry add requests           # add to [tool.poetry.dependencies]
poetry add pytest --group dev # add to [tool.poetry.group.dev.dependencies]
poetry add "fastapi>=0.100"   # with version constraint

# ── INSTALLING ────────────────────────────────────────────────────────────────

poetry install                # install all deps from poetry.lock
poetry install --without dev  # production only

# ── MANAGING THE ENVIRONMENT ──────────────────────────────────────────────────

poetry shell                  # activate the venv
poetry run python myscript.py # run command in the venv
poetry env info               # show env details

# ── UPDATING ─────────────────────────────────────────────────────────────────

poetry update                 # update all deps (respects constraints)
poetry update requests        # update a specific package
poetry lock                   # regenerate lock file without installing

# ── BUILDING & PUBLISHING ─────────────────────────────────────────────────────

poetry build                  # build sdist and wheel into dist/
poetry publish                # upload to PyPI (requires credentials)
"""
print(poetry_commands)

## pyproject.toml Anatomy

The `pyproject.toml` file is now the standard for Python project configuration, replacing `setup.py`, `setup.cfg`, and `MANIFEST.in`.

In [ ]:
%%writefile pyproject.toml
[tool.poetry]
name = "myproject"
version = "0.1.0"
description = "A sample Python project"
authors = ["Ada Lovelace <ada@example.com>"]
license = "MIT"
readme = "README.md"
homepage = "https://github.com/ada/myproject"
repository = "https://github.com/ada/myproject"
documentation = "https://myproject.readthedocs.io"
keywords = ["sample", "python", "project"]
classifiers = [
    "Programming Language :: Python :: 3",
    "License :: OSI Approved :: MIT License",
    "Operating System :: OS Independent",
]

[tool.poetry.dependencies]
python = "^3.11"
requests = "^2.28"
fastapi = "^0.100"
uvicorn = {extras = ["standard"], version = "^0.23"}
python-dotenv = "^1.0"

[tool.poetry.group.dev.dependencies]
pytest = "^7.4"
pytest-cov = "^4.1"
black = "^23.0"
flake8 = "^6.0"
mypy = "^1.5"

[tool.poetry.scripts]
myproject = "myproject.cli:main"  # entry point

[build-system]
requires = ["poetry-core"]
build-backend = "poetry.core.masonry.api"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.mypy]
strict = true
python_version = "3.11"

## The Lock File — Reproducible Installs

Both Poetry (`poetry.lock`) and pip-tools (`requirements.txt`) produce lock files. A lock file:

- Records the **exact version** of every package (including transitive deps)
- Records the **hash** of each package wheel for security
- Guarantees that `poetry install` on any machine gives the **exact same environment**

**Golden rule**: commit `pyproject.toml` AND `poetry.lock` to version control.

In [ ]:
# Demonstrate Poetry lock file concept
lock_example = """
# poetry.lock excerpt — you never edit this manually
[[package]]
name = "requests"
version = "2.31.0"                  # exact version pinned
description = "Python HTTP for Humans."
category = "main"
optional = false
python-versions = ">=3.7"

[package.dependencies]
certifi = ">=2017.4.17"
charset-normalizer = ">=2,<4"
idna = ">=2.5,<4"
urllib3 = ">=1.21.1,<3"

[package.extras]
socks = ["PySocks (>=1.5.6,!=1.5.7)"]
use-chardet-on-py3 = ["chardet (>=3.0.2,<6)"]

[[package]]
name = "certifi"
version = "2023.11.17"             # transitive dep — also pinned
...

[metadata]
lock-version = "2.0"
python-versions = "^3.11"
content-hash = "sha256:abc123..."  # hash of pyproject.toml
"""
print(lock_example)

## venv vs conda vs Poetry — Comparison

| Feature | venv + pip | conda | Poetry |
|---|---|---|---|
| Python management | No | Yes | No (uses system Python) |
| Non-Python deps | No | Yes (C libs, etc.) | No |
| Lock file | pip-tools | `environment.yml` | `poetry.lock` |
| Publishing to PyPI | manual | No | Built-in |
| Learning curve | Low | Medium | Medium |
| Best for | Simple projects | Data science | Libraries/apps |

### venv in VS Code

VS Code detects `.venv` automatically. You can also:
- `Ctrl+Shift+P` → "Python: Select Interpreter" → choose `.venv/bin/python`
- The integrated terminal will auto-activate the venv

In [ ]:
# conda commands for reference (data science teams)
conda_commands = """
# Create environment from scratch
conda create -n myenv python=3.11

# Activate
conda activate myenv

# Install packages (mix of conda and pip)
conda install numpy pandas matplotlib
pip install somepackage-not-on-conda

# Export environment
conda env export > environment.yml

# Recreate from file
conda env create -f environment.yml

# List environments
conda env list

# Deactivate
conda deactivate
"""

# environment.yml example
env_yml = """
# environment.yml
name: myenv
channels:
  - defaults
  - conda-forge
dependencies:
  - python=3.11
  - numpy=1.25.2
  - pandas=2.1.0
  - matplotlib=3.8.0
  - pip:
    - somepackage==1.0.0  # pip-only packages
"""

print("Conda commands:")
print(conda_commands)
print("environment.yml example:")
print(env_yml)

## Practice Exercises

**Exercise 1 — pip-tools workflow**
Create a `requirements.in` file with these abstract dependencies:
```
flask>=2.3
sqlalchemy>=2.0
python-dotenv>=1.0
```
Run `pip-compile` to generate `requirements.txt`, then inspect it to see what transitive dependencies were added.

**Exercise 2 — pyproject.toml**
Write a complete `pyproject.toml` for a hypothetical project called `weather-cli` that:
- Uses Python 3.11+
- Has `requests` and `rich` as production dependencies
- Has `pytest` and `black` as dev dependencies
- Defines a CLI entry point `weather = "weather_cli.main:run"`

**Exercise 3 — Dependency groups**
With Poetry installed, create a new directory and run `poetry init` interactively (or write the `pyproject.toml` manually), then add one dependency to each group: `[dependencies]`, `[group.dev.dependencies]`, and `[group.docs.dependencies]`. Explain why separating these groups matters for deployment.